# MES · QMS · FDC 온톨로지: 기존 테이블만 사용

앞에서 적재한 MES 3개·QMS 8개 테이블과 FDC 시계열을 연결하는 워크숍용 노트북입니다. 원본 MES에 재접속하거나 데이터 노트북을 실행하지 않습니다. **데이터 테이블을 새로 만들거나 수정하지 않습니다.**

## 참가자별 준비

1. 본인 작업 영역의 QMS·MES Lakehouse가 스키마 활성화 상태이며 `Tables → dbo` 아래에 원본 테이블이 있는지 확인합니다. 생성 창의 **Lakehouse schemas** 옵션으로 설정합니다.
2. QMS·FDC·MES 데이터는 동일한 MES 이력 기준으로 준비합니다. MES는 `mes_equipment`, `mes_lot`, `mes_process_result`만 필요합니다. 제품·자재·BOM 추가 수집은 필요하지 않습니다.
3. **본인의 MES Lakehouse를 이 노트북의 기본 Lakehouse로 Attach**합니다. QMS Lakehouse와 FDC Eventhouse/KQL Database도 같은 작업 영역에 있어야 합니다.
4. **4번 셀**의 `QMS_LAKEHOUSE_ID`, `KQL_DATABASE_ID`에 본인의 ID를 입력합니다. KQL Database ID는 Eventhouse ID와 다릅니다. 현재 코드는 같은 작업 영역의 정확한 이름 입력도 지원하며, 중복 이름이면 중단합니다.
5. 위에서부터 실행하고 원본 검사 전 출력되는 저장소 이름·ID가 맞는지 확인합니다.

Fabric 용량, 원본 읽기 권한, 온톨로지 생성 권한(Contributor 이상)과 Ontology Preview 테넌트 설정이 필요합니다. 인증은 노트북 실행자 신원을 사용합니다. Lakehouse 정적 바인딩에는 관리형 Delta 테이블을 사용하며, OneLake 보안과 column mapping 활성화 원본에는 제한이 있습니다.

## 이 노트북이 하는 일

- 기존 MES·QMS 테이블로 **11개 엔터티 유형과 18개 관계 유형**을 정의합니다.
- FDC `fdc_sensor_reading`을 `eqp_id` 기준으로 기존 **Equipment**의 시계열에 직접 바인딩합니다.
- 센서 종류·단위·측정값·상태·시각을 함께 노출합니다. 센서 규격은 원본 검사에만 사용하며 온톨로지 속성으로 복사하지 않습니다.
- 원본 키·참조·시간·센서 채널·중복을 확인하고, 온톨로지 게시 후 Graph 새로 고침을 기다립니다.

**센서 보조 테이블과 별도 센서 엔터티는 생성하지 않습니다.** 원본 13개 테이블은 그대로 유지됩니다. 용량·Lakehouse·Eventhouse·데이터 에이전트도 만들지 않습니다.

## 데모 검증 범위

MES·QMS 관계 질문은 같은 폴더의 비즈니스 질문 가이드를 따릅니다. **Equipment 직접 시계열 바인딩의 Fabric 실행·질의 결과는 별도 리허설이 필요합니다.** 같은 설비·시각에 센서 여러 개가 있으므로 단일 센서 필터와 행별 값 대응을 확인하기 전에는 FDC 복합 질문을 데모 성공 항목으로 취급하지 않습니다.

패키지는 `microsoft-fabric-api==0.1.0b24`를 사용합니다. 정의 등록과 실제 질문 응답 성공은 별개입니다.

In [ ]:
%pip install --quiet microsoft-fabric-api==0.1.0b24 'azure-identity>=1.17,<2'


In [ ]:
QMS_LAKEHOUSE_ID = "<YOUR QMS LAKEHOUSE ID>"
KQL_DATABASE_ID = "<YOUR KQL DATABASE ID>"


## 1. 원본 테이블 계약과 모델

기존 테이블로 11개 엔터티 유형과 18개 관계 유형을 만듭니다.

| 원본 | 엔터티 |
|---|---|
| MES Lakehouse 3개 테이블 | Equipment, ProductionLot, ProcessRun |
| QMS Lakehouse 8개 테이블 | Inspector, InspectionSpec, Inspection, Measurement, DefectCode, IncomingInspection, Nonconformance, Disposition |
| FDC Eventhouse 판독값 | 별도 엔터티 없이 Equipment의 시계열 속성으로 연결 |

Equipment의 정적 키는 `eqp_id`입니다. 시계열에는 `sensor_code`, `unit`, `reading_ts`, `value`, `status`, `run_status`를 함께 바인딩합니다. 센서 종류와 단위를 먼저 제한한 뒤 측정값을 비교하며, 같은 시각의 여러 센서 값을 합치지 않습니다.

센서 정상 범위·경보 한계는 온톨로지에 바인딩하지 않습니다. 경보 질문은 원본에 기록된 `status`를 사용합니다. 규격 대비 수치 계산은 Eventhouse에서 별도로 확인합니다. 제품·자재·BOM 엔터티는 추가하지 않습니다.

In [ ]:
SOURCE_SCHEMAS = {
  "mes_equipment": {
    "eqp_id": "STRING",
    "eqp_type": "STRING"
  },
  "mes_lot": {
    "lot_id": "STRING",
    "product_code": "STRING"
  },
  "mes_process_result": {
    "id": "INT",
    "lot_id": "STRING",
    "eqp_id": "STRING",
    "step_code": "STRING",
    "in_time": "TIMESTAMP",
    "out_time": "TIMESTAMP"
  },
  "qms_defect_code": {
    "defect_code": "STRING",
    "defect_name_ko": "STRING",
    "defect_name_en": "STRING",
    "mes_defect_code": "STRING",
    "defect_category": "STRING",
    "severity": "STRING",
    "severity_score": "INT",
    "typical_step_codes": "STRING",
    "standard_cause_ko": "STRING",
    "standard_action_ko": "STRING",
    "is_active": "BOOLEAN"
  },
  "qms_inspection_spec": {
    "spec_id": "STRING",
    "product_code": "STRING",
    "product_name": "STRING",
    "step_code": "STRING",
    "step_name": "STRING",
    "characteristic_code": "STRING",
    "characteristic_name_ko": "STRING",
    "measurement_type": "STRING",
    "unit": "STRING",
    "target_value": "DOUBLE",
    "lsl": "DOUBLE",
    "usl": "DOUBLE",
    "cpk_target": "DOUBLE",
    "sampling_method": "STRING",
    "sample_size": "INT",
    "inspection_frequency": "STRING",
    "control_method_ko": "STRING",
    "spec_version": "STRING",
    "effective_from": "DATE",
    "is_active": "BOOLEAN"
  },
  "qms_inspector": {
    "inspector_id": "STRING",
    "inspector_name": "STRING",
    "team_ko": "STRING",
    "shift_code": "STRING",
    "qualification_level": "STRING",
    "certified_characteristics": "STRING",
    "certified_from": "DATE",
    "certified_until": "DATE",
    "is_active": "BOOLEAN"
  },
  "qms_inspection": {
    "inspection_id": "STRING",
    "inspection_type": "STRING",
    "lot_id": "STRING",
    "product_code": "STRING",
    "product_name": "STRING",
    "step_code": "STRING",
    "step_name": "STRING",
    "eqp_id": "STRING",
    "mes_process_result_id": "INT",
    "inspector_id": "STRING",
    "inspection_datetime": "TIMESTAMP",
    "sample_size": "INT",
    "inspected_wafer_qty": "INT",
    "judgment": "STRING",
    "judgment_basis_ko": "STRING",
    "defect_found_qty": "INT",
    "measurement_count": "INT",
    "has_nonconformance": "BOOLEAN",
    "remark_ko": "STRING"
  },
  "qms_measurement": {
    "measurement_id": "STRING",
    "inspection_id": "STRING",
    "spec_id": "STRING",
    "lot_id": "STRING",
    "product_code": "STRING",
    "step_code": "STRING",
    "characteristic_code": "STRING",
    "characteristic_name_ko": "STRING",
    "sample_no": "INT",
    "site_no": "INT",
    "measured_value": "DOUBLE",
    "unit": "STRING",
    "target_value": "DOUBLE",
    "lsl": "DOUBLE",
    "usl": "DOUBLE",
    "deviation_pct": "DOUBLE",
    "is_out_of_spec": "BOOLEAN",
    "judgment": "STRING",
    "metrology_eqp_id": "STRING",
    "measured_by": "STRING",
    "measured_at": "TIMESTAMP"
  },
  "qms_incoming_inspection": {
    "iqc_id": "STRING",
    "material_code": "STRING",
    "material_name": "STRING",
    "supplier_code": "STRING",
    "supplier_name_ko": "STRING",
    "supplier_lot_no": "STRING",
    "receipt_date": "DATE",
    "received_qty": "DOUBLE",
    "uom": "STRING",
    "sample_size": "INT",
    "inspection_items_ko": "STRING",
    "judgment": "STRING",
    "defect_code": "STRING",
    "coa_received": "BOOLEAN",
    "coa_conformance": "STRING",
    "inspector_id": "STRING",
    "inspection_date": "DATE",
    "remark_ko": "STRING"
  },
  "qms_nonconformance": {
    "ncr_id": "STRING",
    "ncr_source": "STRING",
    "inspection_id": "STRING",
    "iqc_id": "STRING",
    "lot_id": "STRING",
    "product_code": "STRING",
    "product_name": "STRING",
    "step_code": "STRING",
    "step_name": "STRING",
    "material_code": "STRING",
    "eqp_id": "STRING",
    "defect_code": "STRING",
    "mes_defect_code": "STRING",
    "severity": "STRING",
    "affected_qty": "INT",
    "detected_date": "DATE",
    "root_cause_category": "STRING",
    "root_cause_ko": "STRING",
    "immediate_action_ko": "STRING",
    "owner_dept_ko": "STRING",
    "owner_name": "STRING",
    "status": "STRING",
    "due_date": "DATE",
    "closed_date": "DATE",
    "estimated_cost_krw": "BIGINT"
  },
  "qms_disposition": {
    "disposition_id": "STRING",
    "ncr_id": "STRING",
    "lot_id": "STRING",
    "product_code": "STRING",
    "step_code": "STRING",
    "disposition_type": "STRING",
    "disposition_qty": "INT",
    "decision_date": "DATE",
    "decision_body_ko": "STRING",
    "approver_name": "STRING",
    "approval_status": "STRING",
    "rework_step_code": "STRING",
    "rework_result": "STRING",
    "scrap_cost_krw": "BIGINT",
    "effectiveness_check_date": "DATE",
    "effectiveness_result": "STRING",
    "reason_ko": "STRING"
  },
  "fdc_sensor_spec": {
    "sensor_code": "STRING",
    "sensor_name_ko": "STRING",
    "eqp_type": "STRING",
    "unit": "STRING",
    "normal_min": "DOUBLE",
    "normal_max": "DOUBLE",
    "alarm_min": "DOUBLE",
    "alarm_max": "DOUBLE",
    "sample_interval_sec": "INT",
    "is_active": "BOOLEAN"
  },
  "fdc_sensor_reading": {
    "reading_ts": "TIMESTAMP",
    "eqp_id": "STRING",
    "eqp_type": "STRING",
    "step_code": "STRING",
    "run_status": "STRING",
    "sensor_code": "STRING",
    "value": "DOUBLE",
    "unit": "STRING",
    "status": "STRING"
  }
}


In [ ]:
import base64
import datetime as dt
import hashlib
import json
import math
import re
import uuid
from urllib.parse import urlsplit

from microsoft_fabric_api.generated.ontology.models import (
    CreateOntologyRequest, OntologyDefinition, OntologyDefinitionPart,
    UpdateOntologyDefinitionRequest,
)


class OntologyError(RuntimeError):
    """An explicit failure in preparing or publishing the ontology."""


SDK_VERSION = "0.1.0b24"
MODEL_TAG = "iq-series-mes-qms-fdc-v2"
TELEMETRY_SCHEMA = {
    "sensor_code": "STRING", "unit": "STRING", "reading_ts": "TIMESTAMP",
    "value": "DOUBLE", "status": "STRING", "run_status": "STRING",
}
ENTITY_SPECS = [
    ("Equipment", "Eqp", "mes_equipment", ("eqp_id",), "MES production equipment, not QMS metrology instruments. Telemetry contains multiple sensors at the same timestamp: filter sensor_code and unit before reading or aggregating value. Telemetry is synthetic, not proof of causality."),
    ("ProductionLot", "Lot", "mes_lot", ("lot_id",), "A production lot. Operational details remain in the original MES."),
    ("ProcessRun", "Run", "mes_process_result", ("id",), "One MES process execution. Correlate sensor timestamps using in_time <= reading_ts < out_time on the same eqp_id."),
    ("Inspector", "Person", "qms_inspector", ("inspector_id",), "QMS personnel and qualification information."),
    ("InspectionSpec", "Spec", "qms_inspection_spec", ("spec_id",), "Product- and process-specific quality limits. Join measurements by spec_id, never characteristic_code alone."),
    ("Inspection", "Insp", "qms_inspection", ("inspection_id",), "Quality inspection: IPQC, IPQC-RT, OQC, PCS or EQV. Not every inspection belongs to a production lot."),
    ("Measurement", "Meas", "qms_measurement", ("measurement_id",), "QMS metrology values. metrology_eqp_id is not a MES/FDC production equipment identifier."),
    ("DefectCode", "Def", "qms_defect_code", ("defect_code",), "QMS detailed defect taxonomy; mes_defect_code is a category mapping, not a process execution key."),
    ("IncomingInspection", "Iqc", "qms_incoming_inspection", ("iqc_id",), "Incoming material quality. Supplier lots are not proof of actual production consumption."),
    ("Nonconformance", "Ncr", "qms_nonconformance", ("ncr_id",), "Quality issue from an inspection, incoming inspection, or customer complaint. Material-to-lot traces in the sample are synthetic."),
    ("Disposition", "Disp", "qms_disposition", ("disposition_id",), "Quality disposition and approval state. A special-acceptance decision alone does not prove shipment approval."),
]
RELATION_SPECS = [
    ("LotHasRuns", "ProductionLot", "ProcessRun", "mes_process_result", ("lot_id",), ("id",)),
    ("EquipmentHasRuns", "Equipment", "ProcessRun", "mes_process_result", ("eqp_id",), ("id",)),
    ("InspectorDoesInspection", "Inspector", "Inspection", "qms_inspection", ("inspector_id",), ("inspection_id",)),
    ("InspectorRecordsMeasure", "Inspector", "Measurement", "qms_measurement", ("measured_by",), ("measurement_id",)),
    ("InspectorDoesIncoming", "Inspector", "IncomingInspection", "qms_incoming_inspection", ("inspector_id",), ("iqc_id",)),
    ("SpecGovernsMeasurement", "InspectionSpec", "Measurement", "qms_measurement", ("spec_id",), ("measurement_id",)),
    ("InspectionHasMeasures", "Inspection", "Measurement", "qms_measurement", ("inspection_id",), ("measurement_id",)),
    ("DefectInIncoming", "DefectCode", "IncomingInspection", "qms_incoming_inspection", ("defect_code",), ("iqc_id",)),
    ("DefectInNcr", "DefectCode", "Nonconformance", "qms_nonconformance", ("defect_code",), ("ncr_id",)),
    ("InspectionHasNcr", "Inspection", "Nonconformance", "qms_nonconformance", ("inspection_id",), ("ncr_id",)),
    ("IncomingHasNcr", "IncomingInspection", "Nonconformance", "qms_nonconformance", ("iqc_id",), ("ncr_id",)),
    ("NcrHasDisposition", "Nonconformance", "Disposition", "qms_disposition", ("ncr_id",), ("disposition_id",)),
    ("RunHasInspection", "ProcessRun", "Inspection", "qms_inspection", ("mes_process_result_id",), ("inspection_id",)),
    ("LotHasInspection", "ProductionLot", "Inspection", "qms_inspection", ("lot_id",), ("inspection_id",)),
    ("EquipmentHasInspection", "Equipment", "Inspection", "qms_inspection", ("eqp_id",), ("inspection_id",)),
    ("LotHasNcr", "ProductionLot", "Nonconformance", "qms_nonconformance", ("lot_id",), ("ncr_id",)),
    ("EquipmentHasNcr", "Equipment", "Nonconformance", "qms_nonconformance", ("eqp_id",), ("ncr_id",)),
    ("LotHasDisposition", "ProductionLot", "Disposition", "qms_disposition", ("lot_id",), ("disposition_id",)),
]
VALUE_TYPES = {
    "STRING": "String", "INT": "BigInt", "BIGINT": "BigInt",
    "DOUBLE": "Double", "BOOLEAN": "Boolean", "DATE": "DateTime",
    "TIMESTAMP": "DateTime",
}


def checked_guid(value, label):
    if not isinstance(value, str):
        raise OntologyError(f"{label} must be an item/workspace GUID.")
    try:
        return str(uuid.UUID(value))
    except ValueError:
        raise OntologyError(f"{label} must be a valid item/workspace GUID.") from None


def checked_name(value, label, allow_empty=False, maximum=100):
    if allow_empty and value == "":
        return value
    if not isinstance(value, str) or not re.fullmatch(r"[A-Za-z][A-Za-z0-9_]{0," + str(maximum - 1) + "}", value):
        raise OntologyError(f"{label} must be a simple identifier starting with a letter.")
    return value


def validate_settings(settings):
    out = dict(settings)
    for field in ("workspace_id", "mes_lakehouse_id", "qms_lakehouse_id", "eventhouse_id", "kql_database_id"):
        out[field] = checked_guid(out.get(field), field)
    if len({out[field] for field in ("mes_lakehouse_id", "qms_lakehouse_id", "eventhouse_id", "kql_database_id")}) != 4:
        raise OntologyError("MES Lakehouse, QMS Lakehouse, Eventhouse, and KQL database must be distinct items.")
    for field in ("ontology_id", "graph_model_id"):
        value = out.get(field, "")
        out[field] = checked_guid(value, field) if value else ""
    for field in ("mes_schema", "qms_schema"):
        out[field] = checked_name(out.get(field, ""), field, allow_empty=True)
    out["ontology_name"] = checked_name(out.get("ontology_name"), "ontology_name", maximum=99)
    database = out.get("database_name")
    if not isinstance(database, str) or not database.strip() or database != database.strip():
        raise OntologyError("database_name must match the existing KQL database name.")
    uri = out.get("kusto_uri")
    if not isinstance(uri, str):
        raise OntologyError("kusto_uri must be the existing Eventhouse HTTPS Query URI.")
    parsed = urlsplit(uri)
    if (
        parsed.scheme != "https" or not parsed.hostname
        or not parsed.hostname.endswith((".kusto.fabric.microsoft.com", ".kusto.windows.net"))
        or parsed.username or parsed.password or parsed.query or parsed.fragment
        or parsed.path not in ("", "/") or parsed.port not in (None, 443)
    ):
        raise OntologyError("kusto_uri must be a Fabric/Kusto HTTPS Query URI without credentials or path.")
    out["kusto_uri"] = uri.rstrip("/")
    if type(out.get("allow_definition_update", False)) is not bool:
        raise OntologyError("allow_definition_update must be boolean.")
    return out


def stable_id(kind, name):
    number = int.from_bytes(hashlib.sha256(f"{MODEL_TAG}/{kind}/{name}".encode()).digest()[:8], "big")
    minimum, maximum = (10001, 2**48) if kind == "entity" else (100001, 2**63)
    return str(minimum + number % (maximum - minimum))


def binding_id(label):
    return str(uuid.uuid5(uuid.NAMESPACE_URL, f"{MODEL_TAG}/{label}"))


def property_name(prefix, column):
    name = f"{prefix}_{column}"
    if len(name) > 26:
        name = name[:19] + "_" + hashlib.sha256(name.encode()).hexdigest()[:6]
    return name


def source_columns(table):
    return SOURCE_SCHEMAS[table]


def lakehouse_source(settings, table):
    side = "qms" if table.startswith("qms_") else "mes"
    result = {
        "sourceType": "LakehouseTable", "workspaceId": settings["workspace_id"],
        "itemId": settings[f"{side}_lakehouse_id"], "sourceTableName": table,
    }
    if settings[f"{side}_schema"]:
        result["sourceSchema"] = settings[f"{side}_schema"]
    return result


def build_definition(settings):
    settings = validate_settings(settings)
    parts, entities, properties = [], {}, {}
    used_ids, used_names = set(), set()

    def identity(kind, label):
        value = stable_id(kind, label)
        if value in used_ids:
            raise OntologyError("A deterministic model ID collided; change the model namespace explicitly.")
        used_ids.add(value)
        return value

    def add(path, payload):
        encoded = base64.b64encode(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode()).decode()
        parts.append(OntologyDefinitionPart(path=path, payload=encoded, payload_type="InlineBase64"))

    def prop(entity_name, prefix, column, kind):
        name = property_name(prefix, column)
        if name in used_names:
            raise OntologyError("Property names must be globally unique in this model.")
        used_names.add(name)
        pid = identity("property", f"{entity_name}/{column}")
        properties[(entity_name, column)] = pid
        return {
            "id": pid, "name": name, "valueType": VALUE_TYPES[kind],
            "semanticEnrichment": {"description": f"{entity_name}: source column {column}."},
        }

    def binding(entity, label, table_source, mappings, timestamp=None):
        bid = binding_id(f"{entity}/{label}")
        body = {
            "dataBindingType": "TimeSeries" if timestamp else "NonTimeSeries",
            "sourceTableProperties": table_source,
            "propertyBindings": [
                {"sourceColumnName": column, "targetPropertyId": properties[(entity, column)]}
                for column in mappings
            ],
        }
        if timestamp:
            body["timestampColumnName"] = timestamp
        add(f"EntityTypes/{entities[entity]['id']}/DataBindings/{bid}.json", {
            "id": bid, "dataBindingConfiguration": body,
        })

    add(".platform", {"metadata": {"type": "Ontology", "displayName": settings["ontology_name"]}})
    add("definition.json", {})
    for entity_name, prefix, table, keys, description in ENTITY_SPECS:
        entity = {
            "id": identity("entity", entity_name), "namespace": "usertypes",
            "namespaceType": "Custom", "name": entity_name, "visibility": "Visible",
            "properties": [prop(entity_name, prefix, column, kind) for column, kind in source_columns(table).items()],
            "timeseriesProperties": [],
            "semanticEnrichment": {
                "description": description,
                "customAttributes": {"managedBy": MODEL_TAG, "sourceTable": table},
            },
        }
        entity["entityIdParts"] = [properties[(entity_name, column)] for column in keys]
        entity["displayNamePropertyId"] = properties[(entity_name, keys[-1])]
        if entity_name == "Equipment":
            entity["timeseriesProperties"] = [
                prop(entity_name, prefix, column, kind)
                for column, kind in TELEMETRY_SCHEMA.items()
            ]
        entities[entity_name] = entity
        add(f"EntityTypes/{entity['id']}/definition.json", entity)
        binding(entity_name, "static", lakehouse_source(settings, table), source_columns(table))

    binding("Equipment", "telemetry", {
        "sourceType": "KustoTable", "workspaceId": settings["workspace_id"],
        "itemId": settings["eventhouse_id"], "clusterUri": settings["kusto_uri"],
        "databaseName": settings["database_name"], "sourceTableName": "fdc_sensor_reading",
    }, ("eqp_id", *TELEMETRY_SCHEMA), timestamp="reading_ts")

    for name, source, target, table, source_cols, target_cols in RELATION_SPECS:
        rid = identity("relationship", name)
        add(f"RelationshipTypes/{rid}/definition.json", {
            "id": rid, "namespace": "usertypes", "namespaceType": "Custom", "name": name,
            "source": {"entityTypeId": entities[source]["id"]},
            "target": {"entityTypeId": entities[target]["id"]},
            "semanticEnrichment": {"description": f"{source} to {target}, using existing keys in {table}. Optional null references do not identify an entity."},
        })
        bid = binding_id(f"relationship/{name}")
        add(f"RelationshipTypes/{rid}/Contextualizations/{bid}.json", {
            "id": bid, "dataBindingTable": lakehouse_source(settings, table),
            "sourceKeyRefBindings": [
                {"sourceColumnName": column, "targetPropertyId": pid}
                for column, pid in zip(source_cols, entities[source]["entityIdParts"])
            ],
            "targetKeyRefBindings": [
                {"sourceColumnName": column, "targetPropertyId": pid}
                for column, pid in zip(target_cols, entities[target]["entityIdParts"])
            ],
        })
    definition = OntologyDefinition(parts=parts)
    decoded_parts(definition)
    return definition


def decoded_parts(definition):
    raw = definition.serialize() if isinstance(definition, OntologyDefinition) else definition
    if not isinstance(raw, dict) or not isinstance(raw.get("parts"), list):
        raise OntologyError("Ontology definition must contain a parts list.")
    decoded = {}
    for part in raw["parts"]:
        if not isinstance(part, dict) or part.get("payloadType") != "InlineBase64":
            raise OntologyError("Unsupported ontology definition part.")
        name = part.get("path")
        if not isinstance(name, str) or name in decoded:
            raise OntologyError("Missing or duplicate ontology definition part path.")
        try:
            decoded[name] = json.loads(base64.b64decode(part["payload"], validate=True))
        except (ValueError, KeyError, UnicodeDecodeError):
            raise OntologyError(f"Invalid JSON/base64 definition part: {name}.") from None
    return decoded


def canonical_definition(definition):
    unordered = {"properties", "timeseriesProperties", "untypedProperties", "propertyBindings"}

    def normalized(value, field=None):
        if isinstance(value, dict):
            return {key: normalized(item, key) for key, item in sorted(value.items()) if item is not None and item != "" and item != [] and item != {}}
        if isinstance(value, list):
            items = [normalized(item) for item in value]
            return sorted(items, key=lambda item: json.dumps(item, sort_keys=True)) if field in unordered else items
        return value
    parts = decoded_parts(definition)
    parts.pop(".platform", None)
    for payload in parts.values():
        if isinstance(payload, dict):
            payload.pop("$schema", None)
    return json.dumps(normalized(parts), sort_keys=True, ensure_ascii=False, separators=(",", ":"))


def create_request(settings, definition):
    return CreateOntologyRequest(
        display_name=settings["ontology_name"],
        description=f"Manufacturing context from existing MES/QMS/FDC sources. Managed by {MODEL_TAG}.",
        definition=definition,
    ).serialize()


def utc_value(value):
    if isinstance(value, str):
        try:
            value = dt.datetime.fromisoformat(value.replace("Z", "+00:00"))
        except ValueError:
            raise OntologyError("A source timestamp is not ISO-compatible.") from None
    if not isinstance(value, dt.datetime):
        raise OntologyError("A source timestamp is missing or invalid.")
    return value.replace(tzinfo=dt.timezone.utc) if value.tzinfo is None else value.astimezone(dt.timezone.utc)


def validate_source_rows(tables):
    indexed = {}
    for entity, _, table, keys, _ in ENTITY_SPECS:
        rows = tables.get(table)
        if not isinstance(rows, list) or not rows:
            raise OntologyError(f"{table} has no rows; run the source notebooks first.")
        expected = set(source_columns(table))
        index = {}
        for row in rows:
            if not expected <= set(row):
                raise OntologyError(f"{table} is missing required source columns.")
            values = tuple(row[column] for column in keys)
            if any(type(value) not in (str, int) or value == "" for value in values) or values in index:
                raise OntologyError(f"{table} has null, invalid or duplicate entity identifiers.")
            index[values] = row
        indexed[entity] = index
    for name, source, target, table, source_cols, target_cols in RELATION_SPECS:
        for row in tables[table]:
            for entity, columns in ((source, source_cols), (target, target_cols)):
                key = tuple(row[column] for column in columns)
                if any(value is None for value in key):
                    continue
                if key not in indexed[entity]:
                    raise OntologyError(f"{name}: non-null reference does not resolve to {entity}.")
    runs = indexed["ProcessRun"]
    anchor = max(utc_value(row["out_time"]) for row in runs.values())
    for row in runs.values():
        if utc_value(row["out_time"]) < utc_value(row["in_time"]):
            raise OntologyError("MES process execution has a reversed time interval.")
    for inspection in tables["qms_inspection"]:
        key = inspection["mes_process_result_id"]
        if key is None:
            continue
        run = runs[(key,)]
        if any(inspection[field] != run[field] for field in ("lot_id", "eqp_id", "step_code")):
            raise OntologyError("QMS inspection and MES execution identifiers describe different production context.")
        when = utc_value(inspection["inspection_datetime"])

        if not utc_value(run["out_time"]) <= when <= anchor:            raise OntologyError("QMS and MES timestamps do not share the expected sample time axis.")

In [ ]:
import time
from urllib.parse import parse_qsl, urlencode, urlsplit, urlunsplit

from azure.core.exceptions import ServiceRequestError, ServiceResponseError
from azure.core.rest import HttpRequest


FABRIC_BASE = "https://api.fabric.microsoft.com/v1"


class FabricRequestError(OntologyError):
    def __init__(self, status, code):
        self.status = status
        self.code = code
        super().__init__(f"Fabric HTTP {status}: {code}. Check source access, tenant settings and permissions.")


def error_code(payload):
    if not isinstance(payload, dict):
        return "UnspecifiedError"
    nested = payload.get("error")
    value = payload.get("errorCode") or (nested.get("errorCode") or nested.get("code") if isinstance(nested, dict) else None)
    return value if isinstance(value, str) and re.fullmatch(r"[A-Za-z0-9_.-]{1,120}", value) else "UnspecifiedError"


class FabricApi:
    """Use the real SDK pipeline, but poll Fabric's documented LRO/job contracts explicitly."""

    def __init__(self, client, timeout_seconds=1800, sleep=time.sleep, clock=time.monotonic):
        self.client = client
        self.timeout_seconds = timeout_seconds
        self.sleep = sleep
        self.clock = clock

    def url(self, value):
        if not isinstance(value, str) or value.startswith("//"):
            raise OntologyError("An API URL is invalid.")
        url = FABRIC_BASE + value if value.startswith("/") else value
        parsed = urlsplit(url)
        if (
            parsed.scheme != "https" or parsed.netloc != "api.fabric.microsoft.com"
            or not parsed.path.startswith("/v1/") or parsed.fragment
            or "\\" in parsed.path or ".." in parsed.path.split("/")
        ):
            raise OntologyError("Refusing to send a Fabric token to an unexpected API URL.")
        return url

    def pause(self, headers, deadline, default=2):
        raw = headers.get("retry-after", str(default))
        try:
            delay = max(float(raw), 0.05)
        except (TypeError, ValueError):
            raise OntologyError("Fabric returned an invalid Retry-After value.") from None
        if not math.isfinite(delay) or self.clock() + delay >= deadline:
            raise OntologyError("Fabric operation exceeded its deadline. Completion is unknown; do not blindly recreate the item.")
        self.sleep(delay)

    def response(self, method, path, body=None, deadline=None):
        deadline = self.clock() + self.timeout_seconds if deadline is None else deadline
        url = self.url(path)
        for attempt in range(6):
            if self.clock() >= deadline:
                raise OntologyError("Fabric request deadline exceeded.")
            request = HttpRequest(method, url, json=body) if body is not None else HttpRequest(method, url)
            try:
                response = self.client.send_request(request, logging_enable=False)
            except (ServiceRequestError, ServiceResponseError):
                raise OntologyError(
                    f"Fabric {method} transport failed. A submitted operation may still exist; inspect the item before retrying."
                ) from None
            try:
                status = response.status_code
                headers = {key.lower(): value for key, value in response.headers.items()}
                try:
                    payload = response.json() if response.content else None
                except ValueError:
                    raise OntologyError(f"Fabric returned non-JSON content (HTTP {status}).") from None
            finally:
                response.close()
            if status == 429 and attempt < 5:
                self.pause(headers, deadline)
                continue
            if status not in (200, 201, 202, 204):
                raise FabricRequestError(status, error_code(payload))
            return status, payload, headers
        raise OntologyError("Fabric throttling retry limit exceeded.")

    def request(self, method, path, body=None):
        status, payload, _ = self.response(method, path, body)
        if status == 202:
            raise OntologyError("This request requires explicit long-running-operation handling.")
        return payload

    def operation(self, method, path, body=None, result=False):
        deadline = self.clock() + self.timeout_seconds
        status, payload, headers = self.response(method, path, body, deadline)
        if status != 202:
            if result and payload is None:
                raise OntologyError("Fabric completed a result-bearing operation without a result.")
            return payload if result else None
        identifier = checked_guid(headers.get("x-ms-operation-id"), "Fabric operation ID")
        state_path = f"/operations/{identifier}"
        while True:
            self.pause(headers, deadline)
            _, state, headers = self.response("GET", state_path, deadline=deadline)
            if not isinstance(state, dict):
                raise OntologyError("Fabric operation status is missing.")
            status = state.get("status")
            if status == "Succeeded":
                if result:
                    _, payload, _ = self.response("GET", state_path + "/result", deadline=deadline)
                    if payload is None:
                        raise OntologyError("Fabric operation result is empty.")
                    return payload
                return None
            if status in ("Failed", "Cancelled"):
                raise OntologyError(f"Fabric operation {identifier}: {status} ({error_code(state)}).")
            if status not in ("NotStarted", "Running"):
                raise OntologyError(f"Unrecognized Fabric operation status for {identifier}.")

    def list_all(self, path, collection="value"):
        first = self.url(path)
        current, visited, rows = first, set(), []
        while current:
            current = self.url(current)
            if current in visited:
                raise OntologyError("Fabric pagination repeated a continuation URL.")
            visited.add(current)
            page = self.request("GET", current)
            if not isinstance(page, dict) or not isinstance(page.get(collection), list):
                raise OntologyError(f"Fabric list response is missing its {collection} array.")
            rows.extend(page[collection])
            current = page.get("continuationUri")
            if not current and page.get("continuationToken"):
                parsed = urlsplit(first)
                query = dict(parse_qsl(parsed.query))
                query["continuationToken"] = page["continuationToken"]
                current = urlunsplit((parsed.scheme, parsed.netloc, parsed.path, urlencode(query), ""))
        return rows


def fetch_definition(api, workspace_id, ontology_id):
    body = api.operation(
        "POST", f"/workspaces/{workspace_id}/ontologies/{ontology_id}/getDefinition", result=True,
    )
    if not isinstance(body, dict) or "definition" not in body:
        raise OntologyError("Fabric did not return an ontology definition.")
    decoded_parts(body["definition"])
    return body["definition"]


def plan_publication(api, settings, definition):
    settings = validate_settings(settings)
    workspace = settings["workspace_id"]
    if settings["ontology_id"]:
        item = api.request("GET", f"/workspaces/{workspace}/ontologies/{settings['ontology_id']}")
        if item.get("displayName") != settings["ontology_name"]:
            raise OntologyError("The configured ontology ID belongs to a differently named ontology.")
    else:
        matches = [
            item for item in api.list_all(f"/workspaces/{workspace}/ontologies")
            if item.get("displayName") == settings["ontology_name"]
        ]
        if len(matches) > 1:
            raise OntologyError("More than one ontology has the requested name; specify its ID explicitly.")
        item = matches[0] if matches else None
    if item is None:
        return {"action": "create", "id": None, "base_signature": None}
    identifier = checked_guid(item.get("id"), "ontology ID")
    previous = fetch_definition(api, workspace, identifier)
    base_signature = canonical_definition(previous)
    desired = canonical_definition(definition)
    if base_signature == desired:
        return {"action": "reuse", "id": identifier, "base_signature": base_signature}
    if not settings.get("allow_definition_update", False):
        raise OntologyError(
            "An ontology with a different definition already exists. "
            "Inspect it before explicitly enabling ALLOW_DEFINITION_UPDATE."
        )
    entity_parts = [
        body for path, body in decoded_parts(previous).items()
        if path.startswith("EntityTypes/") and path.endswith("/definition.json")
    ]
    if not entity_parts or any(body.get("semanticEnrichment", {}).get("customAttributes", {}).get("managedBy") != MODEL_TAG for body in entity_parts):
        raise OntologyError("Refusing to replace an ontology not owned by this notebook model.")
    return {"action": "update", "id": identifier, "base_signature": base_signature}


def publish_definition(api, settings, definition, plan):
    workspace = settings["workspace_id"]
    identifier = plan["id"]
    if plan["action"] == "create":
        try:
            item = api.operation(
                "POST", f"/workspaces/{workspace}/ontologies",
                create_request(settings, definition), result=True,
            )
        except FabricRequestError as exc:
            if exc.code != "ItemDisplayNameAlreadyInUse":
                raise
            replacement = plan_publication(api, settings, definition)
            if replacement["action"] != "reuse":
                raise OntologyError("Concurrent ontology creation did not produce the expected definition.") from None
            identifier = replacement["id"]
        else:
            identifier = checked_guid(item.get("id"), "created ontology ID")
    else:
        current = fetch_definition(api, workspace, identifier)
        if canonical_definition(current) != plan["base_signature"]:
            raise OntologyError("Ontology changed after preflight. Refusing to overwrite concurrent edits.")
        if plan["action"] == "update":
            request = UpdateOntologyDefinitionRequest(definition=definition).serialize()
            api.operation(
                "POST", f"/workspaces/{workspace}/ontologies/{identifier}/updateDefinition?updateMetadata=false",
                request, result=False,
            )
    saved = fetch_definition(api, workspace, identifier)
    if canonical_definition(saved) != canonical_definition(definition):
        raise OntologyError(f"Ontology {identifier} was saved but its returned definition differs from the requested model.")
    print(f"Ontology {plan['action']}: {identifier}")
    return identifier


class GraphJobError(OntologyError):
    def __init__(self, identifier, status, code):
        self.status = status
        super().__init__(f"Graph job {identifier}: {status} ({code}).")


def wait_graph_job(api, location, graph_id):
    path = urlsplit(api.url(location)).path.strip("/").split("/")
    if len(path) != 8 or path[:2] != ["v1", "workspaces"] or path[3:7] != ["items", graph_id, "jobs", "instances"]:
        raise OntologyError("Graph job Location does not match the expected graph item.")
    checked_guid(path[2], "job workspace ID")
    checked_guid(path[7], "job instance ID")
    deadline = api.clock() + api.timeout_seconds
    while True:
        _, job, headers = api.response("GET", location, deadline=deadline)
        if not isinstance(job, dict) or job.get("itemId") != graph_id:
            raise OntologyError("Graph refresh job belongs to an unexpected item.")
        status = job.get("status")
        if status == "Completed":
            return job
        if status in ("Failed", "Cancelled", "Deduped"):
            raise GraphJobError(path[7], status, error_code(job.get("failureReason")))
        if status not in ("NotStarted", "InProgress"):
            raise OntologyError("Graph job returned an unrecognized status.")
        api.pause(headers, deadline)


def checked_item(item, identifier, item_type, workspace):
    if (
        not isinstance(item, dict) or item.get("type") != item_type
        or checked_guid(item.get("id"), "returned item ID") != identifier
        or checked_guid(item.get("workspaceId"), "returned workspace ID") != workspace
    ):
        raise OntologyError(f"The returned metadata does not match the configured {item_type}.")
    return item


def verify_storage_sources(api, settings):
    workspace = settings["workspace_id"]
    for side in ("mes", "qms"):
        identifier = settings[f"{side}_lakehouse_id"]
        checked_item(
            api.request("GET", f"/workspaces/{workspace}/lakehouses/{identifier}"),
            identifier, "Lakehouse", workspace,
        )
    database = checked_item(
        api.request("GET", f"/workspaces/{workspace}/kqlDatabases/{settings['kql_database_id']}"),
        settings["kql_database_id"], "KQLDatabase", workspace,
    )
    properties = database.get("properties", {})
    if (
        properties.get("parentEventhouseItemId") != settings["eventhouse_id"]
        or database.get("displayName") != settings["database_name"]
        or not isinstance(properties.get("queryServiceUri"), str)
        or properties["queryServiceUri"].rstrip("/") != settings["kusto_uri"]
    ):
        raise OntologyError("KQL database name, Query URI, or parent Eventhouse differs from the supplied configuration.")
    eventhouse = checked_item(
        api.request("GET", f"/workspaces/{workspace}/eventhouses/{settings['eventhouse_id']}"),
        settings["eventhouse_id"], "Eventhouse", workspace,
    )
    properties = eventhouse.get("properties", {})
    if (
        settings["kql_database_id"] not in properties.get("databasesItemIds", [])
        or not isinstance(properties.get("queryServiceUri"), str)
        or properties["queryServiceUri"].rstrip("/") != settings["kusto_uri"]
    ):
        raise OntologyError("Eventhouse metadata does not confirm the configured KQL database and Query URI.")


def resolve_graph(api, settings, ontology_id):
    workspace = settings["workspace_id"]
    identifier = settings.get("graph_model_id")
    if not identifier:
        expected_name = f"{settings['ontology_name']}_graph_{ontology_id.replace('-', '')}"
        deadline = api.clock() + min(api.timeout_seconds, 300)
        while True:
            matches = [
                item for item in api.list_all(f"/workspaces/{workspace}/graphModels")
                if item.get("displayName") == expected_name
            ]
            if len(matches) > 1:
                raise OntologyError("More than one Graph matches the exact generated name. Specify GRAPH_MODEL_ID.")
            if matches:
                identifier = checked_guid(matches[0].get("id"), "graph model ID")
                print("Graph selected by the exact Microsoft sample naming convention; this is not a parent-link API.")
                break
            if api.clock() + 2 >= deadline:
                raise OntologyError(
                    f"Ontology {ontology_id} is saved, but its Graph could not be identified. "
                    "Set GRAPH_MODEL_ID from this ontology's Graph; no arbitrary Graph was selected."
                )
            api.pause({}, deadline)
    item = api.request("GET", f"/workspaces/{workspace}/graphModels/{identifier}")
    checked_item(item, identifier, "GraphModel", workspace)
    return identifier


def refresh_ontology_graph(api, settings, ontology_id):
    workspace = settings["workspace_id"]
    graph_id = resolve_graph(api, settings, ontology_id)
    base = f"/workspaces/{workspace}/items/{graph_id}/jobs/instances"
    for attempt in range(2):
        active = [
            job for job in api.list_all(base)
            if job.get("status") in ("NotStarted", "InProgress")
        ]
        if len(active) > 1:
            raise OntologyError("Multiple active Graph jobs are present; wait for them before rerunning.")
        if active:
            job_id = checked_guid(active[0].get("id"), "active graph job ID")
            wait_graph_job(api, FABRIC_BASE + base + "/" + job_id, graph_id)
        try:
            status, _, headers = api.response(
                "POST", f"/workspaces/{workspace}/graphModels/{graph_id}/jobs/refreshGraph/instances",
            )
        except FabricRequestError as exc:
            if exc.code == "RefreshAlreadyInProgress" and attempt == 0:
                continue
            raise
        if status == 200:
            return {"ontology_id": ontology_id, "graph_id": graph_id, "status": "Completed"}
        if status != 202 or not headers.get("location"):
            raise OntologyError("Graph refresh returned neither synchronous completion nor an asynchronous job Location.")
        location = api.url(headers["location"])
        if not location.startswith(FABRIC_BASE + base + "/"):
            raise OntologyError("Graph refresh Location points to an unexpected workspace/item.")
        api.pause(headers, api.clock() + api.timeout_seconds, default=0)
        try:
            job = wait_graph_job(api, location, graph_id)
        except GraphJobError as exc:
            if exc.status == "Deduped" and attempt == 0:
                continue
            raise
        return {"ontology_id": ontology_id, "graph_id": graph_id, "job_id": job.get("id"), "status": "Completed"}
    raise OntologyError("Graph refresh could not complete without a concurrent-job conflict.")


In [ ]:
def validate_schema_table(row, schema, tables_root, shortcuts):
    name = row.get("name")
    if not isinstance(name, str) or not name or any(char in name for char in "/\\%") or name in (".", ".."):
        raise OntologyError("Invalid schema table name.")
    if row.get("schema_name") != schema or row.get("data_source_format") != "DELTA":
        raise OntologyError(f"{name}: schema or Delta format mismatch.")
    if row.get("table_type") not in (None, "MANAGED", "Managed"):
        raise OntologyError(f"{name}: external or unsupported table type.")
    location = row.get("storage_location")
    expected = f"{tables_root.rstrip('/')}/{schema}/{name}"
    if not isinstance(location, str) or location.rstrip("/") != expected:
        raise OntologyError(f"{name}: storage is outside its registered Lakehouse schema path.")
    relative = f"Tables/{schema}/{name}"
    for shortcut in shortcuts:
        if not isinstance(shortcut.get("path"), str) or not isinstance(shortcut.get("name"), str):
            raise OntologyError("Cannot verify shortcut metadata.")
        path = shortcut["path"].strip("/") + "/" + shortcut["name"]
        if relative == path or relative.startswith(path + "/"):
            raise OntologyError(f"{name}: table or parent schema is a Shortcut.")
    return location.rstrip("/")


def lakehouse_table_catalog(api, settings, side):
    import requests
    import notebookutils

    workspace = settings["workspace_id"]
    item_id = settings[f"{side}_lakehouse_id"]
    schema = settings[f"{side}_schema"]
    metadata = api.request("GET", f"/workspaces/{workspace}/lakehouses/{item_id}")
    actual_schema = metadata.get("properties", {}).get("defaultSchema")
    if not actual_schema:
        if schema:
            raise OntologyError(f"{side}: a schema was configured for a non-schema Lakehouse.")
        return api.list_all(f"/workspaces/{workspace}/lakehouses/{item_id}/tables", collection="data")
    if not schema:
        raise OntologyError(f"{side}: specify a schema; defaultSchema is {actual_schema}.")
    root = f"https://onelake.dfs.fabric.microsoft.com/{workspace}/{item_id}/Tables"
    if metadata.get("properties", {}).get("oneLakeTablesPath", "").rstrip("/") != root:
        raise OntologyError(f"{side}: unexpected OneLake tables root.")
    shortcuts = api.list_all(f"/workspaces/{workspace}/items/{item_id}/shortcuts")
    endpoint = f"https://onelake.table.fabric.microsoft.com/delta/{workspace}/{item_id}/api/2.1/unity-catalog/tables"
    params = {"catalog_name": metadata["displayName"] + ".Lakehouse", "schema_name": schema}
    catalog = []
    seen = set()
    for page_number in range(100):
        if AUTH_MODE == "notebook":
            token = notebookutils.credentials.getToken("storage")
        else:
            from azure.identity import DefaultAzureCredential
            with DefaultAzureCredential() as credential:
                token = credential.get_token("https://storage.azure.com/.default").token
        response = requests.get(endpoint, params=params, headers={"Authorization": "Bearer " + token}, timeout=60, allow_redirects=False)
        if response.status_code != 200:
            raise OntologyError(f"OneLake table metadata HTTP {response.status_code}; verify source permissions.")
        payload = response.json()
        rows = payload.get("tables")
        if not isinstance(rows, list):
            raise OntologyError("Invalid OneLake table metadata response.")
        required = {name for name in SOURCE_SCHEMAS if name.startswith(side + "_")}
        for row in rows:
            if row.get("name") not in required:
                continue
            location = validate_schema_table(row, schema, root, shortcuts)
            catalog.append({"name": row["name"], "location": location.replace(f"https://onelake.dfs.fabric.microsoft.com/{workspace}/", f"abfss://{workspace}@onelake.dfs.fabric.microsoft.com/", 1), "format": "delta", "localSchemaTable": True})
        next_page = payload.get("next_page_token")
        if not next_page:
            return catalog
        if not isinstance(next_page, str) or next_page in seen:
            raise OntologyError("Invalid or repeated OneLake pagination token.")
        seen.add(next_page)
        params["page_token"] = next_page
    raise OntologyError("OneLake table metadata exceeded the pagination limit.")

In [ ]:
from azure.core.credentials import AccessToken
from microsoft_fabric_api.generated.ontology import FabricOntologyClient


class NotebookCredential:
    """Use the Fabric user's renewable token; JWT decoding is for expiry only."""

    def __init__(self, notebookutils):
        self.notebookutils = notebookutils

    def get_token(self, *scopes, **kwargs):
        token = self.notebookutils.credentials.getToken("pbi")
        try:
            payload = token.split(".")[1]
            claims = json.loads(base64.urlsafe_b64decode(payload + "=" * (-len(payload) % 4)))
            expires = claims["exp"]
        except (IndexError, KeyError, ValueError, UnicodeDecodeError):
            raise OntologyError("NotebookUtils returned a token without a usable expiration claim.") from None
        if type(expires) is not int or expires <= time.time():
            raise OntologyError("NotebookUtils returned an expired token.")
        return AccessToken(token, expires)


def make_api(notebookutils, auth_mode="notebook"):
    if auth_mode == "notebook":
        credential = NotebookCredential(notebookutils)
    elif auth_mode == "default":
        from azure.identity import DefaultAzureCredential
        credential = DefaultAzureCredential()
    else:
        raise OntologyError("AUTH_MODE must be notebook or default.")
    client = FabricOntologyClient(
        credential, endpoint=FABRIC_BASE, retry_total=0, redirect_max=0,
        connection_timeout=30, read_timeout=120, logging_enable=False,
    )
    return FabricApi(client)


def table_path(settings, table):
    source = lakehouse_source(settings, table)
    suffix = f"{source['sourceSchema']}/{table}" if source.get("sourceSchema") else table
    return f"abfss://{source['workspaceId']}@onelake.dfs.fabric.microsoft.com/{source['itemId']}/Tables/{suffix}"


def bounded_rows(frame, maximum, label):
    rows = [row.asDict() for row in frame.limit(maximum + 1).collect()]
    if len(rows) > maximum:
        raise OntologyError(f"{label} exceeds MAX_STATIC_ROWS; this notebook is scoped to the supplied demo datasets.")
    return rows


def assert_frame_schema(frame, expected, label):
    types = dict(frame.dtypes)
    compatible = {
        "STRING": {"string"}, "BOOLEAN": {"boolean", "bool"},
        "INT": {"int", "integer", "bigint", "long"}, "BIGINT": {"int", "integer", "bigint", "long"},
        "DOUBLE": {"double", "float"}, "DATE": {"date"}, "TIMESTAMP": {"timestamp"},
    }
    for column, kind in expected.items():
        if types.get(column) not in compatible[kind]:
            raise OntologyError(f"{label}.{column} is missing or has an incompatible type; expected {kind}.")


def read_lakehouse_sources(api, spark, settings, maximum):
    spark.conf.set("spark.sql.session.timeZone", "UTC")
    catalogs = {side: lakehouse_table_catalog(api, settings, side) for side in ("mes", "qms")}
    tables = {}
    for name, expected in SOURCE_SCHEMAS.items():
        if name.startswith("fdc_"):
            continue
        side = "qms" if name.startswith("qms_") else "mes"
        location = table_path(settings, name)
        candidates = [
            row for row in catalogs[side]
            if isinstance(row.get("location"), str) and row["location"].rstrip("/") == location
        ]
        if len(candidates) != 1 or str(candidates[0].get("format", "")).lower() != "delta":
            raise OntologyError(f"{name}: no unique Delta table at the configured source/schema.")
        candidate = candidates[0]
        if candidate.get("type") != "Managed" and candidate.get("localSchemaTable") is not True:
            raise OntologyError(f"{name}: expected a managed table or a verified local schema table without Shortcuts.")
        detail = spark.sql(f"DESCRIBE DETAIL delta.`{location}`").collect()[0].asDict()
        if detail.get("properties", {}).get("delta.columnMapping.mode", "none") != "none":
            raise OntologyError(f"{name}: ontology binding does not support Delta column mapping.")
        frame = spark.read.format("delta").load(location)
        assert_frame_schema(frame, expected, name)
        expressions = [
            f"CAST(`{column}` AS STRING) AS `{column}`" if kind in ("DATE", "TIMESTAMP") else f"`{column}`"
            for column, kind in expected.items()
        ]
        tables[name] = bounded_rows(frame.selectExpr(*expressions), maximum, name)
        print(f"Read {name}: {len(tables[name])} rows")
    return tables


def kusto_frame(spark, notebookutils, settings, query):
    return (
        spark.read.format("com.microsoft.kusto.spark.synapse.datasource")
        .option("kustoCluster", settings["kusto_uri"])
        .option("kustoDatabase", settings["database_name"])
        .option("kustoQuery", query)
        .option("accessToken", notebookutils.credentials.getToken(settings["kusto_uri"]))
        .load()
    )


def read_sensor_context(spark, notebookutils, settings, tables, maximum):
    validate_source_rows(tables)
    frame = kusto_frame(spark, notebookutils, settings, "fdc_sensor_spec")
    assert_frame_schema(frame, SOURCE_SCHEMAS["fdc_sensor_spec"], "fdc_sensor_spec")
    specifications = bounded_rows(frame, maximum, "fdc_sensor_spec")
    if not specifications:
        raise OntologyError("FDC sensor specifications must be populated first.")
    spec_units = {}
    for spec in specifications:
        key = (spec["eqp_type"], spec["sensor_code"])
        if key in spec_units or any(not isinstance(value, str) or not value for value in (*key, spec["unit"])):
            raise OntologyError("FDC specification keys or units are missing or duplicated.")
        limits = [spec[column] for column in ("alarm_min", "normal_min", "normal_max", "alarm_max")]
        if any(type(value) not in (int, float) or not math.isfinite(value) for value in limits) or not limits[0] <= limits[1] < limits[2] <= limits[3]:
            raise OntologyError("FDC sensor limits are invalid.")
        spec_units[key] = spec["unit"]
    equipment_types = {row["eqp_id"]: row["eqp_type"] for row in tables["mes_equipment"]}
    if any(eqp_type not in {key[0] for key in spec_units} for eqp_type in equipment_types.values()):
        raise OntologyError("MES equipment type has no matching FDC sensor specification.")
    reading_schema = kusto_frame(spark, notebookutils, settings, "fdc_sensor_reading | take 0")
    assert_frame_schema(reading_schema, SOURCE_SCHEMAS["fdc_sensor_reading"], "fdc_sensor_reading")
    channels = bounded_rows(kusto_frame(
        spark, notebookutils, settings,
        "fdc_sensor_reading | summarize by eqp_id, eqp_type, sensor_code, unit",
    ), maximum, "FDC distinct sensor channels")
    if not channels or any(
        row["eqp_id"] not in equipment_types
        or equipment_types[row["eqp_id"]] != row["eqp_type"]
        or (row["eqp_type"], row["sensor_code"]) not in spec_units
        or spec_units[(row["eqp_type"], row["sensor_code"])] != row["unit"]
        for row in channels
    ):
        raise OntologyError("FDC readings have unrecognized equipment, sensor channels or units.")
    duplicates = bounded_rows(kusto_frame(
        spark, notebookutils, settings,
        "fdc_sensor_reading | summarize n=count() by eqp_id, sensor_code, reading_ts | where n > 1 | take 1",
    ), 1, "FDC duplicate check")
    if duplicates:
        raise OntologyError("FDC contains duplicate equipment/sensor/timestamp records. Reconcile the existing FDC data first.")
    print(f"Verified {len(channels)} sensor channels for Equipment telemetry; no data tables were created or modified.")

## 2. 실행 대상 자동 확인과 인증

NotebookUtils에서 현재 작업 영역과 기본 MES Lakehouse를 확인합니다. **4번 셀**에 입력한 QMS Lakehouse와 KQL Database는 같은 작업 영역에서 확인합니다. 이름으로 입력했다면 정확히 일치하는 항목 하나만 선택합니다.

KQL Database의 부모 Eventhouse·Query URI·이름을 가져오고, QMS와 MES가 서로 다른 Lakehouse이며 `dbo`를 사용하는지 검사합니다. 출력된 이름과 ID가 본인 대상인지 확인합니다.

인증 토큰은 실행자 신원으로 얻으며 출력하거나 파일에 저장하지 않습니다. 기본 온톨로지 이름은 `MES_QMS_FDC`입니다. 이전 모델을 보존해 비교할 때만 아래 `ontology_name`을 다른 이름으로 지정합니다.

In [ ]:
import importlib.metadata
import notebookutils

SETTINGS = DEFINITION = PUBLICATION_PLAN = ONTOLOGY_RESULT = None
SAVED_ONTOLOGY_ID = None
AUTH_MODE = "notebook"
MAX_STATIC_ROWS = 50000
REFRESH_GRAPH = True

if importlib.metadata.version("microsoft-fabric-api") != SDK_VERSION:
    raise OntologyError("Install the pinned SDK version before running the remaining cells.")

CONTEXT = notebookutils.runtime.context
WORKSPACE_ID = checked_guid(CONTEXT.get("currentWorkspaceId"), "current workspace ID")
MES_LAKEHOUSE_ID = checked_guid(CONTEXT.get("defaultLakehouseId"), "default MES Lakehouse ID")
if checked_guid(CONTEXT.get("defaultLakehouseWorkspaceId"), "default Lakehouse workspace ID") != WORKSPACE_ID:
    raise OntologyError("Attach your MES Lakehouse from this notebook's workspace as the default.")

# NEW: Resolve QMS_LAKEHOUSE_ID by friendly name if needed.
try:
    QMS_LAKEHOUSE_ID = checked_guid(QMS_LAKEHOUSE_ID, "QMS_LAKEHOUSE_ID")
except OntologyError:
    # Assume QMS_LAKEHOUSE_ID is a lakehouse *name*; resolve it via the Fabric API.
    api_for_lookup = make_api(notebookutils, AUTH_MODE)
    lakehouses = api_for_lookup.list_all(f"/workspaces/{WORKSPACE_ID}/lakehouses")
    matches = [lh for lh in lakehouses if lh.get("displayName") == QMS_LAKEHOUSE_ID]
    if not matches:
        raise OntologyError(
            f"QMS_LAKEHOUSE_ID '{QMS_LAKEHOUSE_ID}' is neither a valid GUID nor a lakehouse name in this workspace."
        )
    if len(matches) > 1:
        raise OntologyError(
            f"Multiple Lakehouses named '{QMS_LAKEHOUSE_ID}' found. Configure QMS_LAKEHOUSE_ID with the Lakehouse GUID instead."
        )
    QMS_LAKEHOUSE_ID = checked_guid(matches[0].get("id"), "QMS_LAKEHOUSE_ID")

# NEW: Resolve KQL_DATABASE_ID by friendly name if needed.
try:
    KQL_DATABASE_ID = checked_guid(KQL_DATABASE_ID, "KQL_DATABASE_ID")
except OntologyError:
    api_for_lookup = make_api(notebookutils, AUTH_MODE)
    databases = api_for_lookup.list_all(f"/workspaces/{WORKSPACE_ID}/kqlDatabases")
    matches = [db for db in databases if db.get("displayName") == KQL_DATABASE_ID]
    if not matches:
        raise OntologyError(
            f"KQL_DATABASE_ID '{KQL_DATABASE_ID}' is neither a valid GUID nor a KQL Database name in this workspace."
        )
    if len(matches) > 1:
        raise OntologyError(
            f"Multiple KQL Databases named '{KQL_DATABASE_ID}' found. Configure KQL_DATABASE_ID with the KQL Database GUID instead."
        )
    KQL_DATABASE_ID = checked_guid(matches[0].get("id"), "KQL_DATABASE_ID")

if MES_LAKEHOUSE_ID == QMS_LAKEHOUSE_ID:
    raise OntologyError("The default Lakehouse must be MES, not the supplied QMS Lakehouse.")

API = make_api(notebookutils, AUTH_MODE)
DATABASE = checked_item(
    API.request("GET", f"/workspaces/{WORKSPACE_ID}/kqlDatabases/{KQL_DATABASE_ID}"),
    KQL_DATABASE_ID, "KQLDatabase", WORKSPACE_ID,
)
DATABASE_PROPERTIES = DATABASE.get("properties", {})
SETTINGS = validate_settings({
    "workspace_id": WORKSPACE_ID,
    "mes_lakehouse_id": MES_LAKEHOUSE_ID,
    "qms_lakehouse_id": QMS_LAKEHOUSE_ID,
    "eventhouse_id": DATABASE_PROPERTIES.get("parentEventhouseItemId"),
    "kql_database_id": KQL_DATABASE_ID,
    "kusto_uri": DATABASE_PROPERTIES.get("queryServiceUri"),
    "database_name": DATABASE.get("displayName"),
    "mes_schema": "dbo",
    "qms_schema": "dbo",
    "ontology_name": "MES_QMS_FDC",
    "ontology_id": "",
    "allow_definition_update": False,
    "graph_model_id": "",
})
verify_storage_sources(API, SETTINGS)
print(f"Workspace: {WORKSPACE_ID}")
for side in ("mes", "qms"):
    identifier = SETTINGS[f"{side}_lakehouse_id"]
    lakehouse = checked_item(
        API.request("GET", f"/workspaces/{WORKSPACE_ID}/lakehouses/{identifier}"),
        identifier, "Lakehouse", WORKSPACE_ID,
    )
    if lakehouse.get("properties", {}).get("defaultSchema") != "dbo":
        raise OntologyError(f"{side}: this workshop requires a schema-enabled Lakehouse with defaultSchema dbo.")
    print(f"{side.upper()} Lakehouse: {lakehouse['displayName']} ({identifier}), schema: dbo")
print(f"KQL Database: {SETTINGS['database_name']} ({KQL_DATABASE_ID})")
print(f"Eventhouse ID: {SETTINGS['eventhouse_id']}")
print(f"Query URI: {SETTINGS['kusto_uri']}")
print(f"Ontology target: {SETTINGS['ontology_name']}")


## 3. 기존 원본 확인

MES·QMS의 스키마·엔터티 키·참조를 검사합니다. 검사와 공정이력이 다른 로트·설비·공정 또는 시간축을 가리키면 중단합니다.

FDC 규격은 메모리에서 센서 종류·단위·한계를 검사하는 데만 사용합니다. 판독값은 스키마·고유 채널·중복 유무를 조회하며 전체 시계열을 복사하지 않습니다. 동일 설비·센서·시각의 중복은 중단하지만, 서로 다른 센서가 같은 시각에 있는 것은 정상입니다.

원본 데이터는 변경하지 않습니다. 이 검사는 바인딩 정의를 준비하기 위한 것이며, Fabric 질의에서 센서별 필터가 정확히 적용되는지까지 증명하지 않습니다.

In [ ]:
TABLES = None
DEFINITION = PUBLICATION_PLAN = None
TABLES = read_lakehouse_sources(API, spark, SETTINGS, MAX_STATIC_ROWS)
read_sensor_context(spark, notebookutils, SETTINGS, TABLES, MAX_STATIC_ROWS)
DEFINITION = build_definition(SETTINGS)
PUBLICATION_PLAN = plan_publication(API, SETTINGS, DEFINITION)
print(f"Definition: {len(ENTITY_SPECS)} entity types, {len(RELATION_SPECS)} relationship types, {len(DEFINITION.parts)} parts")
print(f"Publication action: {PUBLICATION_PLAN['action']}")

## 4. 온톨로지 구성

원본 검사가 성공한 뒤 온톨로지 정의만 게시합니다. **데이터 테이블 생성·저장 단계는 없습니다.**

같은 이름의 온톨로지가 없으면 생성하고, 정의가 같으면 재사용합니다. 기존 정의가 다르거나 동시 편집이 발견되면 중단합니다. 이전 센서 엔터티 모델을 자동으로 덮어쓰지 않습니다. 비교하려면 실행 대상 설정의 `ontology_name`에 다른 이름을 지정하고 원본 검사부터 다시 진행합니다.

시간 초과나 실패 후에는 Fabric에 이미 생성된 항목이 있는지 확인한 뒤 재실행합니다.

In [ ]:
if DEFINITION is None or PUBLICATION_PLAN is None:
    raise OntologyError("Run source preflight successfully before publishing.")
SAVED_ONTOLOGY_ID = publish_definition(API, SETTINGS, DEFINITION, PUBLICATION_PLAN)
ONTOLOGY_ID = SAVED_ONTOLOGY_ID

## 5. Graph 새로 고침

정의가 저장되었다는 사실과 그래프에 데이터가 반영되었다는 사실은 다릅니다.
아래 단계는 현재 작업 영역에서 Fabric이 생성한 해당 온톨로지의 Graph를 찾아 새로 고침을 수행하고,
작업 상태가 `Completed`가 될 때까지 기다립니다. 이미 진행 중인 작업은 재시작하지 않고 기다립니다.

자동으로 하위 Graph를 확정할 수 없으면 임의의 Graph를 선택하지 않고 중단합니다.
이 경우 Fabric 화면에서 온톨로지의 Graph 생성 상태를 확인하세요.
자동 탐색은 Microsoft 예제의 **정확한 Graph 명명 규칙**을 사용하는 방식이며, 소유 관계를 증명하는 API는 아닙니다.
실습은 Graph 새로 고침까지 실행하며, 실패하거나 확인되지 않은 상태를 완료로 표시하지 않습니다.


In [ ]:
if REFRESH_GRAPH:
    ONTOLOGY_RESULT = refresh_ontology_graph(API, SETTINGS, SAVED_ONTOLOGY_ID)
    print(f"Graph refresh completed: {ONTOLOGY_RESULT['graph_id']}")
else:
    ONTOLOGY_RESULT = {"ontology_id": SAVED_ONTOLOGY_ID, "status": "DefinitionOnly"}
    print("Definition saved. Graph refresh was explicitly skipped.")
print(f"Ontology ID: {SAVED_ONTOLOGY_ID}")


## 완성되는 구성과 데모 순서

기존 QMS 8개·MES 3개·FDC 2개 원본 테이블은 그대로 유지합니다. 이 노트북은 온톨로지와 Fabric이 관리하는 Graph만 구성합니다. 파생 데이터 테이블은 만들지 않습니다.

1. Graph 새로 고침 완료 후 MES·QMS 엔터티의 건수와 대표 관계를 조회합니다.
2. 같은 폴더의 [비즈니스 질문 가이드](README-ontology-business-scenarios.md)에서 기본 데모 질문을 실행합니다.
3. FDC는 Equipment 하나·센서 하나·명시한 시간 구간을 조회하여 `sensor_code`, `unit`, `reading_ts`, `value`, `status`가 같은 원본 행에 대응하는지 먼저 확인합니다. 이 직접 바인딩의 실제 질의는 아직 미검증입니다.
4. 센서별 필터와 동일 시각 다중 센서의 구분을 확인한 환경에서만 FDC 확장 질문을 시연합니다. 실패하면 MES·QMS 기본 질문만 사용하고 FDC는 별도 Eventhouse 데모로 분리합니다.

공정별 센서 연결은 같은 설비에서 `in_time <= reading_ts < out_time` 조건을 질의에 적용해야 합니다. 키 관계가 이 시간 조건을 자동 계산하지 않습니다. 센서 규격의 정상 범위·경보 한계는 원본 검사에만 사용하므로 온톨로지에서 규격 대비 이탈량을 질문하지 않습니다.

시연 종료 후 본인이 만든 불필요한 온톨로지만 Fabric에서 정리합니다. 이전 버전 보조 테이블이나 원본 MES·QMS·FDC 테이블은 이 노트북이 삭제하지 않습니다. 재사용 여부를 확인하지 않고 삭제하지 마세요.

## 공식 자료

- [Ontology definition 규격](https://learn.microsoft.com/en-us/rest/api/fabric/articles/item-management/definitions/ontology-definition)
- [정적·시계열 데이터 바인딩](https://learn.microsoft.com/en-us/fabric/iq/ontology/how-to-bind-data)
- [관계 유형과 매핑](https://learn.microsoft.com/en-us/fabric/iq/ontology/how-to-create-relationship-types)
- [Graph 새로 고침](https://learn.microsoft.com/en-us/fabric/iq/ontology/how-to-view-entity-type-details#refresh-the-graph-model)